# 파일과 예외

UTF-8 텍스트 파일을 안전하게 읽고, 예상한 오류만 처리하려면 어떻게 해야 할까요?

## 실행 환경 준비

Python 3.12 표준 라이브러리만 사용합니다. `from pathlib import Path`는 표준 라이브러리에서 경로를 다루는 `Path`라는 이름을 가져옵니다. 모듈 가져오기는 07절에서 자세히 배웁니다. `TemporaryDirectory()`는 실습 전용 폴더를 만들고 그 위치를 반환하므로 개인 파일을 변경하지 않습니다. `/`는 `Path` 뒤에서 사용할 때 폴더와 파일 이름을 연결합니다.

In [1]:
# 실습 전용 임시 폴더에 UTF-8 주문 파일을 만듭니다.
from pathlib import Path
from tempfile import TemporaryDirectory

workspace = TemporaryDirectory(prefix="ch02_05_")
training_dir = Path(workspace.name)
orders_path = training_dir / "orders.txt"
orders_path.write_text("""연필,3
노트,2
""", encoding="utf-8")
print("준비한 파일:", orders_path.name)

준비한 파일: orders.txt


## 기본 실행

`read_items(path)`의 입력은 읽을 파일의 경로입니다. `path.read_text(encoding="utf-8")`가 파일의 바이트를 UTF-8 규칙으로 해석해 하나의 문자열을 반환합니다. 이어서 `text.splitlines()`가 줄바꿈을 기준으로 문자열을 나누어 문자열 목록을 반환합니다. 타입 힌트는 07절에서 배우므로 여기서는 함수의 입력과 반환을 문장으로 확인합니다.

In [2]:
# 파일 본문을 읽어 줄마다 하나의 문자열로 나눕니다.
def read_items(path):
    text = path.read_text(encoding="utf-8")
    return text.splitlines()

items = read_items(orders_path)
print(items)
print("주문 줄 수:", len(items))

['연필,3', '노트,2']
주문 줄 수: 2


## 중간 결과 관찰

읽은 각 줄을 쉼표에서 나누어 상품명과 수량을 확인합니다. `line.split(",")`은 한 문자열을 입력받아 쉼표 기준의 문자열 목록을 반환합니다. 왼쪽의 `name, count_text`는 반환된 두 항목을 차례로 나누어 저장합니다. 파일에서 읽은 `count_text`는 문자열이므로 `int()`로 정수로 바꾼 뒤 `(상품명, 수량)` 튜플을 반환합니다. 반복문으로 각 줄의 반환값을 `parsed` 목록에 추가합니다.

In [3]:
# 쉼표 오른쪽의 수량을 정수로 바꾸어 주문을 구성합니다.
def parse_order(line):
    name, count_text = line.split(",")
    return name, int(count_text)

parsed = []
for line in items:
    parsed.append(parse_order(line))
print(parsed)
assert parsed[0] == ("연필", 3)

[('연필', 3), ('노트', 2)]


## 값 하나를 바꿔 비교

같은 인코딩과 처리 함수를 유지하고 파일 내용의 수량만 바꿉니다. 파일을 다시 읽은 결과도 기본 반복문으로 처리합니다. `parsed[0][1]`은 첫 번째 주문 튜플의 두 번째 값인 수량을 읽습니다.

In [4]:
# 첫 상품의 수량만 3에서 5로 바꾸고 결과를 비교합니다.
orders_path.write_text("""연필,5
노트,2
""", encoding="utf-8")
changed = []
for line in read_items(orders_path):
    changed.append(parse_order(line))
print("변경 전 첫 수량:", parsed[0][1])
print("변경 후 첫 수량:", changed[0][1])
assert changed[0][1] == 5

변경 전 첫 수량: 3
변경 후 첫 수량: 5


## 실패 재현

존재하지 않는 파일 경로를 `read_items()`에 입력하면 파일 읽기 처리에서 `FileNotFoundError`가 발생하며 정상 반환값은 만들어지지 않습니다. `try` 블록이 읽기를 시도하고, `except FileNotFoundError as error`가 그 종류의 예외만 받아 다음 문장을 실행합니다. `else`는 예외가 발생하지 않았을 때 실행되며, 이 실험에서는 예상한 실패가 없었다는 뜻이므로 `AssertionError`를 발생시킵니다.

In [5]:
# 존재하지 않는 파일을 읽을 때 발생하는 예외를 확인합니다.
missing_path = training_dir / "missing.txt"
try:
    read_items(missing_path)
except FileNotFoundError as error:
    print(type(error).__name__, missing_path.name)
else:
    raise AssertionError("FileNotFoundError가 발생해야 합니다.")

FileNotFoundError missing.txt


## 수정

파일 경로를 입력받아 기존 `read_items()`로 처리하되, 파일이 없을 때만 빈 목록을 반환하는 함수를 만듭니다. 정상 파일이면 문자열 목록이 반환되고, 누락 파일이면 `except FileNotFoundError`의 빈 목록이 반환됩니다. 다른 종류의 오류는 이 블록과 일치하지 않으므로 감추지 않고 호출한 곳으로 이어집니다.

In [6]:
# 파일 누락만 빈 목록으로 처리하고 다른 예외는 그대로 둡니다.
def read_items_or_empty(path):
    try:
        return read_items(path)
    except FileNotFoundError:
        return []

result = read_items_or_empty(missing_path)
print("누락 파일 결과:", result)
assert result == []

누락 파일 결과: []
